<a href="https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula06c_mnist_flat_torch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
from tensorflow.keras.datasets import mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print(train_images.shape)
print(train_labels.shape)
print(test_images.shape)
print(test_labels.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [22]:
train_flat_images = train_images.reshape((60000, 28*28))
test_flat_images = test_images.reshape((10000, 28*28))
print(train_flat_images.shape)
print(test_flat_images.shape)

(60000, 784)
(10000, 784)


In [23]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [24]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

train_flat_images_tensor = torch.from_numpy(train_flat_images).float().to(device)
train_labels_tensor = torch.from_numpy(train_labels).to(device)
test_flat_images_tensor = torch.from_numpy(test_flat_images).float().to(device)
test_labels_tensor = torch.from_numpy(test_labels).to(device)

train_dataset = TensorDataset(train_flat_images_tensor, train_labels_tensor)
test_dataset = TensorDataset(test_flat_images_tensor, test_labels_tensor)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [25]:
class BasicTorchNN(nn.Module):
  def __init__(self, num_classes):
    super(BasicTorchNN, self).__init__()
    self.fc1 = nn.Linear(28*28, 512)
    self.fc2 = nn.Linear(512, num_classes)
  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x

model = BasicTorchNN(10).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

epochs = 5
batch_size = 128

for epoch in range(epochs):
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
  print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

with torch.no_grad():
  correct = 0
  total = 0
  for data in test_loader:
    images, labels = data
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
  print(f"Accuracy: {correct / total}")

Epoch 1/5, Loss: 1.0145976543426514
Epoch 2/5, Loss: 0.385921448469162
Epoch 3/5, Loss: 0.32947638630867004
Epoch 4/5, Loss: 0.19865047931671143
Epoch 5/5, Loss: 0.08703279495239258
Accuracy: 0.9459


In [28]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np

class TorchWrappedNN(BaseEstimator, ClassifierMixin):
  def __init__(self, epochs=5, batch_size=128, model_fabric=BasicTorchNN):
    self.epochs = epochs
    self.batch_size = batch_size
    self.model_fabric = model_fabric

  def fit(self, X, y):
    self.labels, ids = np.unique(y, return_inverse=True)

    ytensor = torch.tensor(ids, dtype=torch.long).to(device)
    xtensor = torch.tensor(X, dtype=torch.float32).to(device)
    dataset = TensorDataset(xtensor, ytensor)
    loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

    self.model = self.model_fabric(len(self.labels)).to(device)
    self.optimizer = optim.RMSprop(self.model.parameters(), lr=0.001)
    self.criterion = nn.CrossEntropyLoss()

    for epoch in range(self.epochs):
      for batch in loader:
        inputs, labels = batch
        self.optimizer.zero_grad()
        outputs = self.model(inputs)
        loss = self.criterion(outputs, labels)
        loss.backward()
        self.optimizer.step()
      print(f"Epoch {epoch+1}/{self.epochs}, Loss: {loss.item()}")
    return self

  def predict(self, X):
    with torch.no_grad():
      xtensor = torch.tensor(X, dtype=torch.float32).to(device)
      outputs = self.model(xtensor)
      _, predicted = torch.max(outputs.data, 1)
      return self.labels[predicted.cpu().numpy()]

model = TorchWrappedNN(epochs=5, batch_size=128, model_fabric=BasicTorchNN)
model.fit(train_flat_images, train_labels)
predictions = model.predict(test_flat_images)
print(accuracy_score(test_labels, predictions))

Epoch 1/5, Loss: 0.10360509157180786
Epoch 2/5, Loss: 0.1432926207780838
Epoch 3/5, Loss: 0.09036929160356522
Epoch 4/5, Loss: 0.039264459162950516
Epoch 5/5, Loss: 0.23903687298297882
0.9637


In [32]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

class Divide255(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X / 255.0


class Flatten(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X.reshape((X.shape[0], -1))


pipeline = Pipeline([
    ("flatten", Flatten()),
    ("scaler", Divide255()),
    ("model", TorchWrappedNN())
])

pipeline.fit(train_images, train_labels)
predictions = pipeline.predict(test_images)
print(accuracy_score(test_labels, predictions))

Epoch 1/5, Loss: 0.1706901341676712
Epoch 2/5, Loss: 0.07412569969892502
Epoch 3/5, Loss: 0.04724568501114845
Epoch 4/5, Loss: 0.04140845313668251
Epoch 5/5, Loss: 0.03161649778485298
0.9788


/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
